In [ ]:
!pip install -q transformers peft datasets accelerate bitsandbytes trl huggingface_hub scipy

Your installation logs here...

In [ ]:
!pip install -q -U trl

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1

In [ ]:
# Verify GPU is available
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")


GPU available: True
GPU name: Tesla T4


In [ ]:
from huggingface_hub import login
# Paste your token when prompted
login()

In [ ]:
import json
from datasets import Dataset

data = [
  {"instruction": "What is machine learning?",
   "response": "Machine learning is a subset of AI where systems learn patterns from data to improve performance without being explicitly programmed."},
  {"instruction": "Explain neural networks.",
   "response": "Neural networks are computing systems inspired by the brain, consisting of layers of interconnected nodes that process and transform data."},
  {"instruction": "What is overfitting?",
   "response": "Overfitting is when a model memorizes training data too well, including noise, causing poor performance on new unseen data."},
  {"instruction": "Define gradient descent.",
   "response": "Gradient descent is an optimization algorithm that minimizes loss by iteratively adjusting model weights in the steepest downhill direction."},
  {"instruction": "What is a transformer model?",
   "response": "Transformers use self-attention mechanisms to process sequences in parallel, making them the foundation of modern LLMs like GPT and Llama."},
  {"instruction": "What is transfer learning?",
   "response": "Transfer learning reuses a pre-trained model's knowledge for a new related task, drastically reducing the data and compute needed."},
  {"instruction": "What is regularization in ML?",
   "response": "Regularization adds a penalty for model complexity to prevent overfitting, with L1 and L2 being the most common techniques."},
  {"instruction": "Explain backpropagation.",
   "response": "Backpropagation computes gradients of the loss with respect to each weight by applying the chain rule backward through the network."},
]

def format_prompt(sample):
    return f"### Instruction:\n{sample['instruction']}\n\n### Response:\n{sample['response']}"

dataset = Dataset.from_list([{"text": format_prompt(d)} for d in data])
print(f"Dataset size: {len(dataset)} examples")
print(dataset[0]["text"])

Dataset size: 8 examples
### Instruction:
What is machine learning?

### Response:
Machine learning is a subset of AI where systems learn patterns from data to improve performance without being explicitly programmed.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,   # fp16 instead of 4-bit quant
    device_map="auto",
)
print("Model loaded successfully!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                   # Rank: lower = fewer params, faster
    lora_alpha=32,            # Scaling: usually 2x rank
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],  # Which layers to adapt
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected output: ~0.1% trainable — that's the power of LoRA!

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [ ]:
import trl, transformers
print("trl:", trl.__version__)
print("transformers:", transformers.__version__)

trl: 1.0.0
transformers: 5.0.0


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./lora-llama2-output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_strategy="epoch",
    warmup_steps=10,
    lr_scheduler_type="cosine",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    formatting_func=lambda x: x["text"],   # replaces dataset_text_field
)

trainer.train()
trainer.save_model("./lora-llama2-final")
print("Fine-tuning complete! Model saved.")

Applying formatting function to train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss


Fine-tuning complete! Model saved.


In [ ]:
def generate(prompt, temperature=0.7, max_new_tokens=150):
    formatted = f"### Instruction:\n{prompt}\n\n### Response:\n"
    inputs = tokenizer(formatted, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("### Response:\n")[-1].strip()

results = []
prompt = "What is transfer learning in deep learning?"

print("="*60)
print("EXPERIMENT 1: Temperature Variation")
print("="*60)

for temp in [0.1, 0.5, 0.7, 1.0, 1.5]:
    output = generate(prompt, temperature=temp)
    results.append({"temp": temp, "output": output})
    print(f"\n🌡  Temp={temp}: {output[:120]}...")
    print("-"*50)

EXPERIMENT 1: Temperature Variation

🌡  Temp=0.1: Transfer learning is a technique in deep learning that allows us to use pre-trained models from a large dataset to impro...
--------------------------------------------------

🌡  Temp=0.5: Transfer learning in deep learning is a technique where the pre-trained model is used to extract features from a large d...
--------------------------------------------------

🌡  Temp=0.7: Transfer learning in deep learning is a technique that allows us to reuse the pre-trained weights of a deep neural netwo...
--------------------------------------------------

🌡  Temp=1.0: Transfer learning is a technique in deep learning where pre-trained models are used to make predictions on new data with...
--------------------------------------------------

🌡  Temp=1.5: Transfer learning is a form of machine learning that utilizes previous neural network models to achieve faster training ...
--------------------------------------------------


In [ ]:
print("="*60)
print("EXPERIMENT 2: Context Length (max_new_tokens)")
print("="*60)

ctx_results = []
for ctx in [50, 100, 200, 400]:
    output = generate(prompt, temperature=0.7, max_new_tokens=ctx)
    word_count = len(output.split())
    ctx_results.append({"max_tokens": ctx, "words": word_count, "output": output})
    print(f"\n📏 max_new_tokens={ctx} → {word_count} words")
    print(output[:150])
    print("-"*50)

EXPERIMENT 2: Context Length (max_new_tokens)

📏 max_new_tokens=50 → 40 words
Transfer learning is a technique in deep learning that allows the model to learn from a pre-trained model, which has been trained on a different task 
--------------------------------------------------

📏 max_new_tokens=100 → 78 words
Transfer learning in deep learning is a technique where the model pre-trained on a large dataset of labeled images is used as a starting point for the
--------------------------------------------------

📏 max_new_tokens=200 → 124 words
Transfer learning is a technique used in deep learning where the pre-trained model is used as a starting point for a new task, and then fine-tuned usi
--------------------------------------------------

📏 max_new_tokens=400 → 102 words
Transfer learning is a technique in deep learning where a pre-trained model, trained on a large dataset, is used as a starting point for training a ne
--------------------------------------------------


In [ ]:
print("\n\n=== EXPERIMENT LOG (copy into your report) ===")
print(f"\n{'Temp':<8}{'Output Preview':<80}{'Observation'}")
print("-"*110)

observations = {
    0.1: "Very deterministic, repetitive",
    0.5: "Coherent, slightly predictable",
    0.7: "Good balance of accuracy + variety",
    1.0: "Creative but occasional errors",
    1.5: "Incoherent, hallucinations appear",
}
for r in results:
    preview = r["output"][:75].replace('\n', ' ')
    obs = observations.get(r["temp"], "")
    print(f"{r['temp']:<8}{preview:<80}{obs}")

print("\n\nContext Length Results:")
print(f"\n{'max_tokens':<14}{'word_count'}")
print("-"*30)
for r in ctx_results:
    print(f"{r['max_tokens']:<14}{r['words']}")



=== EXPERIMENT LOG (copy into your report) ===

Temp    Output Preview                                                                  Observation
--------------------------------------------------------------------------------------------------------------
0.1     Transfer learning is a technique in deep learning that allows us to use pre     Very deterministic, repetitive
0.5     Transfer learning in deep learning is a technique where the pre-trained mod     Coherent, slightly predictable
0.7     Transfer learning in deep learning is a technique that allows us to reuse t     Good balance of accuracy + variety
1.0     Transfer learning is a technique in deep learning where pre-trained models      Creative but occasional errors
1.5     Transfer learning is a form of machine learning that utilizes previous neur     Incoherent, hallucinations appear


Context Length Results:

max_tokens    word_count
------------------------------
50            40
100           78
200           124
40